<a href="https://colab.research.google.com/github/JaswanthJavangula/ML_Practise/blob/main/1ML%20JAshu/RandomForest%5BCLASS%26REGRE%5D/Random_Forest_Regression_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import pandas as pd
os.environ["KAGGLE_CACHE"] = "/content/sample_data"

# Set the path to the file you'd like to load
file_path = "cardekho_imputated.csv"

# Load the latest version
df_copy = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "jaswanthjavangula/car-dekho",
  file_path
)
df = df_copy.copy()


/tmp/ipykernel_539/2718960155.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_copy = kagglehub.load_dataset(


100%|██████████| 1.30M/1.30M [00:00<00:00, 2.80MB/s]


In [2]:
df= df.iloc[:,1:]
df

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


DATA CLEANING

handling missing values / handling duplicates

check data type and Understand the data

In [3]:
df.isnull().sum()

,0
car_name,0
brand,0
model,0
vehicle_age,0
km_driven,0
seller_type,0
fuel_type,0
transmission_type,0
mileage,0
engine,0


In [4]:
#remove unwanted columns
df.drop("car_name", axis =1 , inplace = True)
df.drop("brand", axis =1 , inplace = True)

In [5]:
df.head(2)

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.7,796,46.3,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.9,1197,82.0,5,550000


In [6]:
df["model"].unique() # Perform Label Encoding

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [7]:
# different type of features

cat_features = [ i for i in df.columns if df[i].dtypes == "O"]
print("CAT features", len(cat_features))
numerical_features = [ i for i in df.columns if df[i].dtypes != "O"]
print("Numerical features", len(numerical_features))
Discrete_features = [i for i in numerical_features if len(df[i].unique())<=25]
print("Discrete Features", len(Discrete_features))
contious = [ i for i in numerical_features if i not in Discrete_features]
print("Contious features ", len(contious))

CAT features 4
Numerical features 7
Discrete Features 2
Contious features  5


### Feature Engineering

In [8]:
from sklearn.model_selection import train_test_split
X = df.iloc[:,:-1]
y = df.iloc[:,-1:]

In [9]:
X.head(3)

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.7,796,46.3,5
1,Grand,5,20000,Individual,Petrol,Manual,18.9,1197,82.0,5
2,i20,11,60000,Individual,Petrol,Manual,17.0,1197,80.0,5


In [10]:
y.head(2)

,selling_price
0,120000
1,550000


In [11]:
X["model"].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X["model"] = le.fit_transform(X["model"])

In [13]:
X["model"].unique()

array([  7,  54, 118,  38, 100, 117,  96,  88,  97,  32,  29,  24,  16,
        60,  13,  89,  95,  30,  25,  14,  45,  64,  10,  84, 114,  63,
        59,  78,  85,  68,  11,  44,  98,  91,  73,  86,  23,  49,   1,
        19,  41,  62,  61,   0,   4,  92,  42,  83,  90,  58,  79,  39,
         5,  36,  74, 116,   2, 111, 106,  57,  26,  31,  22, 103,  77,
        46,  87,  70, 113,  34,  82,  99, 112,  93,  37, 101,  20,  40,
       115,  47,   3,  81,  33,  12, 107,  51,  28,  18,  65,  80,  94,
        56, 104,  71,  27,  17, 119,  53,  67, 105,  35, 109,  43,   6,
        66,  50,  48, 102, 110, 108,  72,   9,   8,  69,  21,  15,  76,
        52,  75,  55])

In [14]:
X.head(4)

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.3,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.0,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.0,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.1,5


In [15]:
from types import prepare_class
#Column transformer with 3 types of Transformers
num_feature = X.select_dtypes(exclude="object").columns
oh_cols = ["seller_type",	"fuel_type",	"transmission_type"]
label_cols = ["model"]

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_trans = OneHotEncoder()

Preprocessor = ColumnTransformer(
  [
    ("OneHot" , oh_trans, oh_cols),
    ("StandardScaler" , numeric_transformer, num_feature)
    ],remainder= "passthrough"
)

In [16]:
X.head(2) #before

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.7,796,46.3,5
1,54,5,20000,Individual,Petrol,Manual,18.9,1197,82.0,5


In [17]:
X = Preprocessor.fit_transform(X)

In [18]:
pd.DataFrame(X) #after

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-1.519714,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.225693,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.536377,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-1.519714,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,-0.666211,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.508844,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022
15407,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.556082,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444
15408,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.407551,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022
15409,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.426247,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444


### ML Model

In [19]:
X_train,X_test, y_train, y_test = train_test_split(X,y, test_size=0.33, random_state = 10)
X_train.shape, X_test.shape , y_train.shape, y_test.shape

((10325, 17), (5086, 17), (10325, 1), (5086, 1))

In [20]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [21]:
# Metrics function
def eval (true, predicted):
  mae = mean_absolute_error(true, predicted)
  mse = mean_squared_error(true, predicted)
  rmse = np.sqrt(mean_squared_error(true, predicted))
  r2 = r2_score(true, predicted)
  return mae, mse, rmse, r2

In [22]:
# model training
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "RandomForestRegressor": RandomForestRegressor(),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "KNeighborsRegressor": KNeighborsRegressor()
}

for i in range(len(list(models))):
  model =list(models.values())[i]
  model.fit(X_train,y_train)

  #make predictions
  y_train_pred = model.predict(X_train)
  y_test_pred = model.predict(X_test)

  model_train_mae, model_train_mse, model_train_rmse, model_train_R2 = eval(y_train, y_train_pred)
  model_test_mae, model_test_mse, model_test_rmse, model_test_R2 = eval(y_test, y_test_pred)

  print(list(models.keys())[i])

  print("\nModel performance of TRAINING set")
  print(" - Root Mean Squared Error" , model_train_rmse)
  print(" - Mean Absolute Error" , model_train_mae)
  print(" - R2 Score" , model_train_R2)
  print("-"*4)
  print("\nModel performance of TEST set")
  print(" - Root Mean Squared Error" , model_test_rmse)
  print(" - Mean Absolute Error" , model_test_mae)
  print(" - R2 Score" , model_test_R2)
  print("="*35)
  print("\n")

LinearRegression

Model performance of TRAINING set
 - Root Mean Squared Error 478238.26985152665
 - Mean Absolute Error 255597.35411033302
 - R2 Score 0.6663082494289616
----

Model performance of TEST set
 - Root Mean Squared Error 661693.9138510555
 - Mean Absolute Error 270682.0960809155
 - R2 Score 0.5751430601168053


Ridge

Model performance of TRAINING set
 - Root Mean Squared Error 478238.96463241114
 - Mean Absolute Error 255562.2709416849
 - R2 Score 0.6663072798586338
----

Model performance of TEST set
 - Root Mean Squared Error 661716.5219476226
 - Mean Absolute Error 270658.56248276844
 - R2 Score 0.5751140274459174


Lasso

Model performance of TRAINING set
 - Root Mean Squared Error 478238.27039848274
 - Mean Absolute Error 255595.92025229544
 - R2 Score 0.6663082486656822
----

Model performance of TEST set
 - Root Mean Squared Error 661693.9508330036
 - Mean Absolute Error 270680.78523979953
 - R2 Score 0.5751430126264554




/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor

Model performance of TRAINING set
 - Root Mean Squared Error 90210.91229314051
 - Mean Absolute Error 38617.660132071316
 - R2 Score 0.9881266036094779
----

Model performance of TEST set
 - Root Mean Squared Error 445097.42367391
 - Mean Absolute Error 107291.5839160239
 - R2 Score 0.8077622841917544


DecisionTreeRegressor

Model performance of TRAINING set
 - Root Mean Squared Error 17983.68514504987
 - Mean Absolute Error 4194.434221146085
 - R2 Score 0.999528138875814
----

Model performance of TEST set
 - Root Mean Squared Error 505044.76710297406
 - Mean Absolute Error 136665.38701009305
 - R2 Score 0.7524926001587601


KNeighborsRegressor

Model performance of TRAINING set
 - Root Mean Squared Error 249177.2316968906
 - Mean Absolute Error 91603.93220338984
 - R2 Score 0.9094114036436168
----

Model performance of TEST set
 - Root Mean Squared Error 498078.4997904692
 - Mean Absolute Error 121339.42685804168
 - R2 Score 0.7592734304579744




### hyperparameter tuning

In [23]:
knn_params = {"n_neighbors" : [2,5 ,15,25,42,50]}
rf_params = {"max_depth" : [5,6,13,19,None],
             "max_features" : [4,8,12, "auto"],
             "min_samples_split" : [3,5,12,18],
             "n_estimators" : [100,200,300,400,500]}

RScv_models = [
    ("K-Neighbors Regressor" , KNeighborsRegressor(), knn_params),
    ("Random forest Regressor", RandomForestRegressor(), rf_params)
]

In [24]:
from sklearn.model_selection import RandomizedSearchCV
model_params = {}

for model_name , model , params in RScv_models:

  random = RandomizedSearchCV(estimator = model, param_distributions = params,
                            cv = 3, verbose = 2, n_jobs = -1, n_iter = 50)
  random.fit(X_train,y_train)
  model_params[model_name] = random.best_params_

for name in model_params:

  print(f"------ Best Params for {name} ----- ")
  print(model_params[name])
  print("-"*50)


Fitting 3 folds for each of 6 candidates, totalling 18 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 6 is smaller than n_iter=50. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 50 candidates, totalling 150 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
21 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/

------ Best Params for K-Neighbors Regressor ----- 
{'n_neighbors': 2}
--------------------------------------------------
------ Best Params for Random forest Regressor ----- 
{'n_estimators': 300, 'min_samples_split': 3, 'max_features': 8, 'max_depth': 19}
--------------------------------------------------


In [26]:
# model training
models = {
    "RandomForestRegressor": RandomForestRegressor(n_estimators= 300, min_samples_split =3, max_features= 8, max_depth= 19),
    "KNeighborsRegressor": KNeighborsRegressor(n_neighbors = 10, n_jobs=-1)
}

for i in range(len(list(models))):
  model =list(models.values())[i]
  model.fit(X_train,y_train)

  #make predictions
  y_train_pred = model.predict(X_train)
  y_test_pred = model.predict(X_test)

  model_train_mae, model_train_mse, model_train_rmse, model_train_R2 = eval(y_train, y_train_pred)
  model_test_mae, model_test_mse, model_test_rmse, model_test_R2 = eval(y_test, y_test_pred)

  print(list(models.keys())[i])

  print("\nModel performance of TRAINING set")
  print(" - Root Mean Squared Error" , model_train_rmse)
  print(" - Mean Absolute Error" , model_train_mae)
  print(" - R2 Score" , model_train_R2)
  print("-"*4)
  print("\nModel performance of TEST set")
  print(" - Root Mean Squared Error" , model_test_rmse)
  print(" - Mean Absolute Error" , model_test_mae)
  print(" - R2 Score" , model_test_R2)
  print("="*35)
  print("\n")

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor

Model performance of TRAINING set
 - Root Mean Squared Error 105110.13154448607
 - Mean Absolute Error 46840.88257413641
 - R2 Score 0.9838807074854785
----

Model performance of TEST set
 - Root Mean Squared Error 452039.6691125153
 - Mean Absolute Error 104802.27293140542
 - R2 Score 0.8017188024611671


KNeighborsRegressor

Model performance of TRAINING set
 - Root Mean Squared Error 292740.6234963578
 - Mean Absolute Error 104063.18644067796
 - R2 Score 0.8749675318696941
----

Model performance of TEST set
 - Root Mean Squared Error 555444.3563654345
 - Mean Absolute Error 127222.96991742037
 - R2 Score 0.7006291203889847


